# 🧠 How DNNs Break the Curse of Dimensionality: Compositionality Learning

[![arXiv](https://img.shields.io/badge/arXiv-2407.05664-b31b1b.svg)](https://arxiv.org/abs/2407.05664)
[![ICLR 2025](https://img.shields.io/badge/ICLR-2025-blue.svg)](https://iclr.cc/)

**Google Colab Notebook for:**
- **Paper:** "How DNNs Break the Curse of Dimensionality: Compositionality and Symmetry Learning"
- **Authors:** Arthur Jacot, Seok Hoan Choi, Yuxiao Wen (NYU)
- **Conference:** ICLR 2025

---

## 🎯 Research Overview

This notebook demonstrates how Deep Neural Networks learn **compositional functions** f = h∘g to break the curse of dimensionality:
- **g: R^d → R^k** reduces dimensionality (k << d)
- **h: R^k → R^m** operates in reduced space
- **Key insight:** Learning h becomes feasible despite low regularity

## 🏗️ Network Architectures

- **🎢 AccNets (Accordion Networks):** Alternating expansion/contraction layers
- **🏢 Deep Networks:** Standard deep architectures
- **📏 Shallow Networks:** Single hidden layer baselines

# ⚙️ Environment Setup

In [ ]:
# Install dependencies
!pip install torch numpy pandas matplotlib seaborn scipy tqdm pyyaml
!pip install datasets huggingface_hub

# Check GPU availability
import torch
print(f"🚀 CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"📱 GPU Device: {torch.cuda.get_device_name()}")
    print(f"💾 GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    device = torch.device('cuda')
else:
    print("⚠️ Using CPU - experiments will be slower")
    device = torch.device('cpu')

In [ ]:
# Clone the repository with clean implementation
!git clone https://github.com/shc443/CoveringNumber_GB.git
%cd CoveringNumber_GB

# Setup Python path for imports
import sys
import os
sys.path.append('/content/CoveringNumber_GB/cleaning/curse_dimensionality_sim/src')

# Import framework
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import yaml
import logging

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# 📊 Data Loading and Generation

In [ ]:
# Choose data source
USE_HUGGINGFACE_DATA = True  # Set to False to generate fresh data

if USE_HUGGINGFACE_DATA:
    print("📥 Loading pre-generated data from Hugging Face...")
    
    # Load from Hugging Face dataset
    from datasets import load_dataset
    
    try:
        dataset = load_dataset("shc443/MaternKernel_compositionality")
        print(f"✅ Dataset loaded successfully!")
        print(f"📋 Dataset info: {dataset}")
        
        # Extract example data
        sample_data = dataset['train'][0]  # Get first sample
        print(f"📐 Sample data keys: {list(sample_data.keys())}")
        
        # Convert to tensors
        if 'X' in sample_data and 'Y' in sample_data:
            X_sample = torch.tensor(sample_data['X'], device=device)
            Y_sample = torch.tensor(sample_data['Y'], device=device)
            print(f"📊 X shape: {X_sample.shape}, Y shape: {Y_sample.shape}")
        
    except Exception as e:
        print(f"❌ Error loading from Hugging Face: {e}")
        print("🔄 Falling back to fresh data generation...")
        USE_HUGGINGFACE_DATA = False

if not USE_HUGGINGFACE_DATA:
    print("🧪 Generating fresh synthetic data...")
    
    # We'll implement data generation in the next cell

In [ ]:
# Import the clean compositionality framework
try:
    from data.kernels import MaternKernel, CompositionalDataGenerator
    from models import AccordionNet, DeepNet, ShallowNet
    from training.trainer import CompositionTrainer
    from experiments.runner import CompositionExperimentRunner
    from experiments.analysis import CompositionAnalyzer
    from utils.config import ConfigManager
    print("✅ Framework imported successfully!")
    
except ImportError as e:
    print(f"❌ Import error: {e}")
    print("🔧 Setting up framework inline...")
    
    # If imports fail, we'll define key classes inline for Colab
    exec(open('/content/CoveringNumber_GB/cleaning/curse_dimensionality_sim/src/data/kernels.py').read())
    exec(open('/content/CoveringNumber_GB/cleaning/curse_dimensionality_sim/src/models/base_net.py').read())
    exec(open('/content/CoveringNumber_GB/cleaning/curse_dimensionality_sim/src/models/accordion_net.py').read())
    print("✅ Framework loaded inline!")

In [ ]:
# Generate fresh compositional data if needed
if not USE_HUGGINGFACE_DATA:
    print("🔬 Generating compositional datasets...")
    
    # Initialize data generator
    generator = CompositionalDataGenerator(
        d_input=15,
        d_intermediate=3,
        d_output=20,
        n_samples=10000,  # Smaller for Colab
        device=device
    )
    
    # Generate example datasets for different smoothness combinations
    datasets = {}
    smoothness_pairs = [(2.0, 8.0), (8.0, 2.0), (5.0, 5.0)]  # (nu_g, nu_h)
    
    for nu_g, nu_h in smoothness_pairs:
        print(f"📊 Generating data for ν_g={nu_g}, ν_h={nu_h}")
        X, Y = generator.generate_compositional_data(nu_g, nu_h, seed=42)
        datasets[(nu_g, nu_h)] = (X, Y)
    
    print(f"✅ Generated {len(datasets)} datasets")
    
    # Use one dataset for demonstrations
    X_sample, Y_sample = datasets[(2.0, 8.0)]
    
print(f"📐 Data shapes: X={X_sample.shape}, Y={Y_sample.shape}")
print(f"🎯 Device: {X_sample.device}")

# 🏗️ Network Architectures

In [ ]:
# Create different network architectures
print("🏗️ Creating network architectures...")

# Architecture configurations for Colab (smaller than paper for speed)
configs = {
    'accordion': {
        'class': AccordionNet,
        'widths': [15, 400, 50, 20],  # Smaller than paper for Colab
        'L': 3  # Fewer layers for speed
    },
    'deep': {
        'class': DeepNet,
        'widths': [15] + [200] * 6 + [20]  # 6 hidden layers
    },
    'shallow': {
        'class': ShallowNet,
        'widths': [15, 2000, 20]  # Wide shallow network
    }
}

# Create models
models = {}
for name, config in configs.items():
    if name == 'accordion':
        model = config['class'](config['widths'], L=config['L'], loss_type='L2')
    else:
        model = config['class'](config['widths'], loss_type='L2')
    
    model = model.to(device)
    models[name] = model
    
    # Count parameters
    n_params = sum(p.numel() for p in model.parameters())
    print(f"📊 {name.capitalize():10s}: {n_params:,} parameters")

print(f"\n✅ All models created and moved to {device}")

# 🎓 Training Demonstration

In [ ]:
# Quick training demonstration on one dataset
print("🚀 Starting training demonstration...")

# Use the compositional dataset (nu_g=2.0, nu_h=8.0)
X, Y = X_sample, Y_sample
N_train = 8000
N_test = 2000

# Split data
X_train, Y_train = X[:N_train], Y[:N_train]
X_test, Y_test = X[N_train:N_train+N_test], Y[N_train:N_train+N_test]

print(f"📊 Training set: {X_train.shape[0]} samples")
print(f"📊 Test set: {X_test.shape[0]} samples")

# Training configuration (reduced for Colab speed)
training_config = {
    'epochs_per_stage': 300,  # Reduced from 1200
    'lr_scale': 1.0,
    'base_weight_decay': 0.0
}

# Train one model of each type
results = {}

for arch_name in ['accordion', 'deep']:  # Skip shallow for time
    print(f"\n🎯 Training {arch_name.upper()} network...")
    
    # Create fresh model
    model = models[arch_name]
    trainer = CompositionTrainer(model, device)
    
    # Quick single-stage training for demo
    train_loss, test_loss = trainer.train_stage(
        X_train, Y_train, X_test, Y_test,
        lr=0.001,
        weight_decay=0.001,
        epochs=200,
        num_batches=4,
        log_interval=50
    )
    
    # Compute complexity measures
    complexity = model.compute_complexity_bounds(X_train)
    
    results[arch_name] = {
        'train_loss': train_loss,
        'test_loss': test_loss,
        'complexity': complexity
    }
    
    print(f"✅ {arch_name.capitalize()} - Train Loss: {train_loss:.4f}, Test Loss: {test_loss:.4f}")

print("\n🎉 Training demonstration completed!")

# 📈 Analysis and Visualization

In [ ]:
# Compare results across architectures
print("📊 PERFORMANCE COMPARISON")
print("=" * 50)

comparison_df = pd.DataFrame({
    'Architecture': list(results.keys()),
    'Train Loss': [results[arch]['train_loss'] for arch in results.keys()],
    'Test Loss': [results[arch]['test_loss'] for arch in results.keys()],
    'Parameters': [sum(p.numel() for p in models[arch].parameters()) for arch in results.keys()]
})

print(comparison_df.to_string(index=False))

# Complexity comparison
print("\n🧮 COMPLEXITY MEASURES")
print("=" * 50)

for arch_name, result in results.items():
    complexity = result['complexity']
    print(f"\n{arch_name.upper()}:")
    for bound_name, value in complexity.items():
        if isinstance(value, (int, float)):
            print(f"  {bound_name:20s}: {value:.2e}")

In [ ]:
# Create visualizations
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# 1. Performance comparison
architectures = list(results.keys())
test_losses = [results[arch]['test_loss'] for arch in architectures]

axes[0].bar(architectures, test_losses, color=['skyblue', 'lightcoral'])
axes[0].set_ylabel('Test Loss')
axes[0].set_title('Performance Comparison\n(ν_g=2.0, ν_h=8.0)')
axes[0].set_yscale('log')

# 2. Parameter efficiency
param_counts = [sum(p.numel() for p in models[arch].parameters()) for arch in architectures]
efficiency = [test_losses[i] * param_counts[i] for i in range(len(architectures))]

axes[1].bar(architectures, efficiency, color=['lightgreen', 'orange'])
axes[1].set_ylabel('Test Loss × Parameters')
axes[1].set_title('Parameter Efficiency')
axes[1].set_yscale('log')

# 3. Complexity bounds comparison
bounds_to_plot = ['neyshabur_2015', 'bartlett_2017', 'ours_standard_rank']
bound_names = ['Neyshabur 2015', 'Bartlett 2017', 'Our Bound']

x_pos = np.arange(len(bound_names))
width = 0.35

for i, arch in enumerate(architectures):
    if arch in results:
        complexity = results[arch]['complexity']
        bound_values = [complexity.get(bound, 0) for bound in bounds_to_plot]
        axes[2].bar(x_pos + i*width, bound_values, width, label=arch.capitalize())

axes[2].set_ylabel('Complexity Bound')
axes[2].set_title('Complexity Bounds Comparison')
axes[2].set_yscale('log')
axes[2].set_xticks(x_pos + width/2)
axes[2].set_xticklabels(bound_names, rotation=45)
axes[2].legend()

plt.tight_layout()
plt.show()

print("📈 Visualizations generated!")

# 🔬 Mini Parameter Sweep

In [ ]:
# Run a mini parameter sweep suitable for Colab
print("🔬 Starting mini parameter sweep...")

# Colab-friendly parameter space
nu_values = [1.0, 3.0, 6.0, 10.0]  # Reduced set
N_values = [2000, 5000]  # Smaller datasets
architectures = ['accordion', 'deep']  # Skip shallow for time

sweep_results = []
total_experiments = len(nu_values) * len(nu_values) * len(N_values) * len(architectures)

print(f"📊 Running {total_experiments} total experiments...")

with tqdm(total=total_experiments, desc="Parameter Sweep") as pbar:
    for nu_g in nu_values:
        for nu_h in nu_values:
            for N in N_values:
                # Generate data for this parameter combination
                X, Y = generator.generate_compositional_data(nu_g, nu_h, seed=42)
                
                # Split data
                X_train, Y_train = X[:N], Y[:N]
                X_test, Y_test = X[N:N+500], Y[N:N+500]  # Small test set
                
                for arch_name in architectures:
                    # Create fresh model
                    config = configs[arch_name]
                    if arch_name == 'accordion':
                        model = config['class'](config['widths'], L=config['L'], loss_type='L2')
                    else:
                        model = config['class'](config['widths'], loss_type='L2')
                    
                    model = model.to(device)
                    trainer = CompositionTrainer(model, device)
                    
                    # Quick training (reduced epochs for Colab)
                    train_loss, test_loss = trainer.train_stage(
                        X_train, Y_train, X_test, Y_test,
                        lr=0.001,
                        epochs=100,  # Reduced for speed
                        num_batches=2
                    )
                    
                    # Store results
                    sweep_results.append({
                        'nu_g': nu_g,
                        'nu_h': nu_h,
                        'N': N,
                        'architecture': arch_name,
                        'test_loss': test_loss,
                        'train_loss': train_loss
                    })
                    
                    pbar.update(1)
                    
                    # Clear GPU cache to prevent OOM
                    if device.type == 'cuda':
                        torch.cuda.empty_cache()

# Convert to DataFrame
sweep_df = pd.DataFrame(sweep_results)
print(f"\n✅ Parameter sweep completed! {len(sweep_df)} experiments")
print(sweep_df.head())

In [ ]:
# Create heatmaps from mini sweep results
print("🎨 Creating heatmap visualizations...")

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for i, arch in enumerate(['accordion', 'deep']):
    # Filter data for this architecture and largest dataset
    arch_data = sweep_df[
        (sweep_df['architecture'] == arch) & 
        (sweep_df['N'] == max(N_values))
    ]
    
    if len(arch_data) > 0:
        # Create pivot table
        heatmap_data = arch_data.pivot(index='nu_h', columns='nu_g', values='test_loss')
        
        # Plot heatmap
        sns.heatmap(
            heatmap_data, 
            annot=True, 
            fmt='.3f',
            cmap='viridis_r',  # Lower values (better performance) are brighter
            ax=axes[i],
            cbar_kws={'label': 'Test Loss'}
        )
        
        axes[i].set_title(f'{arch.capitalize()} Network\n(N={max(N_values)})')
        axes[i].set_xlabel('ν_g (smoothness of g)')
        axes[i].set_ylabel('ν_h (smoothness of h)')

plt.suptitle('Test Loss vs Smoothness Parameters', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

print("🎯 Key Observations:")
print("• Darker regions = Better performance")
print("• AccNets typically excel when ν_g is small (rough g) and ν_h is large (smooth h)")
print("• This demonstrates compositional learning: g reduces dimension, h operates in reduced space")

# 🧮 Complexity Analysis

In [ ]:
# Analyze relationship between complexity bounds and performance
print("🔍 Analyzing complexity bounds vs empirical performance...")

# Compute complexity for all trained models
complexity_data = []

for _, row in sweep_df.iterrows():
    arch = row['architecture']
    
    # Create model with same architecture
    config = configs[arch]
    if arch == 'accordion':
        model = config['class'](config['widths'], L=config['L'], loss_type='L2')
    else:
        model = config['class'](config['widths'], loss_type='L2')
    model = model.to(device)
    
    # Generate same data
    X, Y = generator.generate_compositional_data(row['nu_g'], row['nu_h'], seed=42)
    X_train = X[:row['N']]
    
    # Compute complexity (using random initialization)
    try:
        complexity = model.compute_complexity_bounds(X_train)
        
        complexity_data.append({
            'nu_g': row['nu_g'],
            'nu_h': row['nu_h'], 
            'N': row['N'],
            'architecture': arch,
            'test_loss': row['test_loss'],
            'lipschitz_product': complexity.get('lipschitz_product', 0),
            'total_norm': complexity.get('total_norm', 0)
        })
    except:
        continue

complexity_df = pd.DataFrame(complexity_data)
print(f"✅ Complexity analysis completed for {len(complexity_df)} experiments")

In [ ]:
# Visualize complexity vs performance
if len(complexity_df) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Plot 1: Lipschitz product vs Test Loss
    for arch in complexity_df['architecture'].unique():
        arch_data = complexity_df[complexity_df['architecture'] == arch]
        axes[0].scatter(
            arch_data['lipschitz_product'], 
            arch_data['test_loss'],
            label=arch.capitalize(),
            alpha=0.7,
            s=60
        )
    
    axes[0].set_xlabel('Lipschitz Product')
    axes[0].set_ylabel('Test Loss')
    axes[0].set_title('Complexity vs Performance')
    axes[0].set_xscale('log')
    axes[0].set_yscale('log')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Plot 2: Total norm vs Test Loss
    for arch in complexity_df['architecture'].unique():
        arch_data = complexity_df[complexity_df['architecture'] == arch]
        axes[1].scatter(
            arch_data['total_norm'], 
            arch_data['test_loss'],
            label=arch.capitalize(),
            alpha=0.7,
            s=60
        )
    
    axes[1].set_xlabel('Total Parameter Norm')
    axes[1].set_ylabel('Test Loss')
    axes[1].set_title('Parameter Norm vs Performance')
    axes[1].set_xscale('log')
    axes[1].set_yscale('log')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.suptitle('Complexity Analysis: Theory vs Practice', fontsize=16)
    plt.tight_layout()
    plt.show()
    
    # Compute correlations
    print("\n📊 CORRELATION ANALYSIS")
    print("=" * 40)
    for arch in complexity_df['architecture'].unique():
        arch_data = complexity_df[complexity_df['architecture'] == arch]
        if len(arch_data) > 1:
            corr_lip = arch_data['lipschitz_product'].corr(arch_data['test_loss'])
            corr_norm = arch_data['total_norm'].corr(arch_data['test_loss'])
            print(f"{arch.capitalize():10s}: Lipschitz corr = {corr_lip:.3f}, Norm corr = {corr_norm:.3f}")
else:
    print("⚠️ No complexity data available for visualization")

# 📚 Learning Curves Analysis

In [ ]:
# Create learning curves showing performance vs dataset size
print("📈 Generating learning curves...")

# Filter results for learning curve analysis
learning_curve_data = sweep_df[
    (sweep_df['nu_g'] == 3.0) &  # Fixed interesting parameter combination
    (sweep_df['nu_h'] == 6.0)
]

if len(learning_curve_data) > 0:
    plt.figure(figsize=(10, 6))
    
    for arch in learning_curve_data['architecture'].unique():
        arch_data = learning_curve_data[learning_curve_data['architecture'] == arch]
        
        plt.plot(
            arch_data['N'], 
            arch_data['test_loss'],
            'o-',
            label=f'{arch.capitalize()}',
            linewidth=2,
            markersize=8
        )
    
    # Add theoretical slope lines
    N_theory = np.array(N_values)
    
    # O(1/√N) theoretical optimal rate
    optimal_rate = 0.1 * (N_theory / N_theory[0]) ** (-0.5)
    plt.plot(N_theory, optimal_rate, 'k--', alpha=0.5, label='O(1/√N) Theory')
    
    # O(1/N^(1/d)) curse of dimensionality rate (for d=15)
    cursed_rate = 0.1 * (N_theory / N_theory[0]) ** (-1/15)
    plt.plot(N_theory, cursed_rate, 'r:', alpha=0.5, label='O(1/N^(1/15)) Cursed')
    
    plt.xscale('log')
    plt.yscale('log')
    plt.xlabel('Dataset Size (N)')
    plt.ylabel('Test Loss')
    plt.title('Learning Curves: Test Loss vs Dataset Size\n(ν_g=3.0, ν_h=6.0)')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Analyze convergence rates
    print("\n📊 CONVERGENCE RATE ANALYSIS")
    print("=" * 40)
    print("Slope of log(test_loss) vs log(N):")
    
    for arch in learning_curve_data['architecture'].unique():
        arch_data = learning_curve_data[learning_curve_data['architecture'] == arch]
        if len(arch_data) > 1:
            # Fit log-log slope
            log_N = np.log(arch_data['N'])
            log_loss = np.log(arch_data['test_loss'])
            slope = np.polyfit(log_N, log_loss, 1)[0]
            print(f"{arch.capitalize():10s}: {slope:.3f} (optimal: -0.5, cursed: {-1/15:.3f})")
            
            if slope < -0.4:
                print(f"  → ✅ Near optimal rate!")
            elif slope > -0.1:
                print(f"  → ⚠️ Suffering from curse of dimensionality")
            else:
                print(f"  → 📊 Intermediate performance")
else:
    print("⚠️ No data available for learning curves")

# 💾 Save Results

In [ ]:
# Save results for download
print("💾 Saving experiment results...")

# Create results directory
os.makedirs('/content/colab_results', exist_ok=True)

# Save sweep results
sweep_df.to_csv('/content/colab_results/mini_parameter_sweep.csv', index=False)
print("✅ Parameter sweep results saved")

# Save complexity analysis if available
if len(complexity_df) > 0:
    complexity_df.to_csv('/content/colab_results/complexity_analysis.csv', index=False)
    print("✅ Complexity analysis saved")

# Save model weights (example)
if 'accordion' in models:
    torch.save({
        'model_state_dict': models['accordion'].state_dict(),
        'config': configs['accordion'],
        'device': str(device)
    }, '/content/colab_results/accordion_model_example.pth')
    print("✅ Example model weights saved")

# Create summary report
summary_report = {
    'experiment_info': {
        'total_experiments': len(sweep_df),
        'architectures_tested': list(sweep_df['architecture'].unique()),
        'nu_range': [float(sweep_df[['nu_g', 'nu_h']].min().min()), 
                    float(sweep_df[['nu_g', 'nu_h']].max().max())],
        'dataset_sizes': list(sweep_df['N'].unique()),
        'device_used': str(device)
    },
    'performance_summary': {}
}

for arch in sweep_df['architecture'].unique():
    arch_data = sweep_df[sweep_df['architecture'] == arch]
    summary_report['performance_summary'][arch] = {
        'mean_test_loss': float(arch_data['test_loss'].mean()),
        'std_test_loss': float(arch_data['test_loss'].std()),
        'best_test_loss': float(arch_data['test_loss'].min()),
        'parameter_count': sum(p.numel() for p in models[arch].parameters())
    }

# Save summary as JSON
import json
with open('/content/colab_results/experiment_summary.json', 'w') as f:
    json.dump(summary_report, f, indent=2)

print("\n📋 EXPERIMENT SUMMARY")
print("=" * 30)
print(f"Total experiments: {summary_report['experiment_info']['total_experiments']}")
print(f"Architectures: {', '.join(summary_report['experiment_info']['architectures_tested'])}")
print(f"Device used: {summary_report['experiment_info']['device_used']}")

print("\n🏆 PERFORMANCE SUMMARY")
for arch, perf in summary_report['performance_summary'].items():
    print(f"{arch.capitalize():12s}: {perf['mean_test_loss']:.4f} ± {perf['std_test_loss']:.4f} (best: {perf['best_test_loss']:.4f})")

print("\n📁 Files saved to /content/colab_results/")
!ls -la /content/colab_results/

# 🎯 Conclusions and Key Insights

## 🔬 Research Findings

This notebook demonstrates the key findings from our ICLR 2025 paper:

### 1. **Compositional Learning**
- Deep networks can efficiently learn f = h∘g compositions
- AccNets explicitly encourage this through architecture
- Performance depends on smoothness of constituent functions

### 2. **Breaking the Curse**
- When g reduces dimensionality effectively, curse is broken
- Learning rate improves from O(N^(-1/d)) to O(N^(-1/2))
- Architecture choice matters for compositional tasks

### 3. **Practical Implications**
- Design networks with explicit bottlenecks for high-dimensional data
- Consider smoothness when choosing architectures
- Complexity bounds can guide architecture selection

## 🚀 Future Directions

- **Real-world applications**: Apply to image, language, scientific data
- **Architecture search**: Automated discovery of optimal AccNet configurations
- **Theory extensions**: Beyond Matérn kernels to natural data distributions

---

**📄 Paper:** [arXiv:2407.05664](https://arxiv.org/abs/2407.05664)  
**💻 Code:** [GitHub Repository](https://github.com/shc443/CoveringNumber_GB)  
**🎓 Authors:** Arthur Jacot, Seok Hoan Choi, Yuxiao Wen (NYU Courant)

# 🔧 Extensions and Experiments

## Run Full Parameter Sweep (Advanced)

For more comprehensive experiments, uncomment and modify the code below:

In [ ]:
# ADVANCED: Full parameter sweep (WARNING: Very computationally intensive!)
# Uncomment to run full experiments from the paper

# # Full parameter space
# nu_values_full = np.arange(0.5, 20.5, 0.5)  # [0.5, 1.0, ..., 20.0]
# N_values_full = [100, 200, 500, 1000, 2000, 5000, 10000, 20000, 50000]
# architectures_full = ['accordion', 'deep', 'shallow']
# 
# # This would take many hours on Colab!
# print(f"⚠️ WARNING: This would run {len(nu_values_full)**2 * len(N_values_full) * len(architectures_full)} experiments")
# print("Estimated time: 20-40+ hours depending on GPU")
# 
# # Uncomment to proceed:
# # full_results = run_full_parameter_sweep(...)

print("ℹ️ Full parameter sweep code available but commented out for safety")
print("ℹ️ For production experiments, use the dedicated experiment runner:")
print("    python experiments/run_compositionality_study.py")

In [ ]:
# GPU memory monitoring and cleanup
if device.type == 'cuda':
    print("🔧 GPU Memory Status:")
    print(f"   Allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
    print(f"   Reserved: {torch.cuda.memory_reserved() / 1e9:.2f} GB")
    print(f"   Max allocated: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")
    
    # Clear cache
    torch.cuda.empty_cache()
    print("🧹 GPU cache cleared")
    
    print(f"   After cleanup - Allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

print("\n🎉 Experiment completed successfully!")
print("\n📁 Download results from /content/colab_results/")
print("💡 For larger experiments, use the full framework locally or on HPC clusters")